In [30]:
# 导入相关包
import numpy as np
import pandas as pd
import pandas.util.testing as tm

import string
PUNCT_TO_REMOVE = string.punctuation

import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

import re
import jieba

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

import lightgbm as lgb
from sklearn.naive_bayes import MultinomialNB,BernoulliNB,GaussianNB
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report,confusion_matrix,f1_score,roc_auc_score,accuracy_score

from sklearn.preprocessing import LabelEncoder

In [31]:
import warnings
warnings.filterwarnings("ignore")

pd.options.display.max_columns = None  # 显示所有列
pd.set_option('display.float_format', lambda x: '%.3f' % x)  # 取消科学计数法

#全部行都能输出
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [32]:
import os

os.getcwd()

'C:\\Users\\user\\Desktop\\练习\\202109\\code'

In [33]:
train = pd.read_csv(r"{}/training.csv".format('../data'))
test = pd.read_csv(r"{}/test-Ans.csv".format('../data'))

In [34]:
train.head()

,CID,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,0,Initmates,Intimate,Intimates,1
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,4,General,Dresses,Dresses,1
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
3,5,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,6,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


In [35]:
test.head()

,CID,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
1,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
2,9,1077,34,Such a fun dress!,I'm 55' and 125 lbs. i ordered the s petite to...,0,General,Dresses,Dresses,1
3,11,1095,39,NaN,This dress is perfection! so pretty and flatte...,2,General Petite,Dresses,Dresses,1
4,13,767,44,Runs big,Bought the black xs to go under the larkspur m...,0,Initmates,Intimate,Intimates,1


In [36]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18789 entries, 0 to 18788
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   CID                      18789 non-null  int64 
 1   Clothing ID              18789 non-null  int64 
 2   Age                      18789 non-null  int64 
 3   Title                    15750 non-null  object
 4   Review Text              18110 non-null  object
 5   Positive Feedback Count  18789 non-null  int64 
 6   Division Name            18775 non-null  object
 7   Department Name          18775 non-null  object
 8   Class Name               18775 non-null  object
 9   Recommended IND          18789 non-null  int64 
dtypes: int64(5), object(5)
memory usage: 1.4+ MB


In [37]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4697 entries, 0 to 4696
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   CID                      4697 non-null   int64 
 1   Clothing ID              4697 non-null   int64 
 2   Age                      4697 non-null   int64 
 3   Title                    3926 non-null   object
 4   Review Text              4531 non-null   object
 5   Positive Feedback Count  4697 non-null   int64 
 6   Division Name            4697 non-null   object
 7   Department Name          4697 non-null   object
 8   Class Name               4697 non-null   object
 9   Recommended IND          4697 non-null   int64 
dtypes: int64(5), object(5)
memory usage: 367.1+ KB


In [38]:
train['text_new']=train['Title'].fillna('')+' '+train['Review Text'].fillna('')+' '+train['Class Name'].fillna('')

In [39]:
train['text_new'][0]

' Absolutely wonderful - silky and sexy and comfortable Intimates'

In [40]:
test['text_new']=test['Title'].fillna('')+' '+test['Review Text'].fillna('')+' '+test['Class Name'].fillna('')

In [42]:
test['text_new'][0]

"My favorite buy! I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments! Pants"

In [43]:
list_train = list(train['text_new'])
list_test = list(test['text_new'])
print(len(list_train))
print(len(list_test))

18789
4697


In [44]:
#英文
chinese = re.compile(u'[\u4e00-\u9fa5]+')
number = re.compile(u'[\u0030-\u0039]')
english = re.compile(u'[\u0041-\u005a\u0061-\u007a]')

def IsEnglish(cks):
    if english.search(cks) and not chinese.search(cks) and not number.search(cks) :
        return True
    else:
        return False

In [45]:
train_content = train.text_new.values.tolist()
train_label = train['Recommended IND'].values.tolist()
test_content = test.text_new.values.tolist()
test_label = test['Recommended IND'].values.tolist()

In [46]:
train_content[0:2]

[' Absolutely wonderful - silky and sexy and comfortable Intimates',
 " Love this dress!  it's sooo pretty.  i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite.  i bought a petite and am 5'8.  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite. Dresses"]

In [49]:
test_content[0:2]

["My favorite buy! I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments! Pants",
 'Flattering shirt This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!! Blouses']

In [51]:
def context(content):
    lis =[]
    for line in content:
        current_segment = jieba.lcut(line)
        if len(current_segment) > 1 and current_segment != '\r\t' :
            current_segment=[segment for segment in current_segment if IsEnglish(segment) and len(segment) > 1]
            #segments=" ".join(current_segment)
            lis.append(current_segment)
    return lis

In [52]:
train_content_S = context(train_content) 
train_content_S[0:2]

Building prefix dict from the default dictionary ...
Loading model from cache C:/Users/user\AppData\Local\Temp\jieba.cache
Loading model cost 0.403 seconds.
Prefix dict has been built successfully.


[['Absolutely',
  'wonderful',
  'silky',
  'and',
  'sexy',
  'and',
  'comfortable',
  'Intimates'],
 ['Love',
  'this',
  'dress',
  'it',
  'sooo',
  'pretty',
  'happened',
  'to',
  'find',
  'it',
  'in',
  'store',
  'and',
  'glad',
  'did',
  'bc',
  'never',
  'would',
  'have',
  'ordered',
  'it',
  'online',
  'bc',
  'it',
  'petite',
  'bought',
  'petite',
  'and',
  'am',
  'love',
  'the',
  'length',
  'on',
  'me',
  'hits',
  'just',
  'little',
  'below',
  'the',
  'knee',
  'would',
  'definitely',
  'be',
  'true',
  'midi',
  'on',
  'someone',
  'who',
  'is',
  'truly',
  'petite',
  'Dresses']]

In [53]:
test_content_S = context(test_content)
test_content_S[0:2]

[['My',
  'favorite',
  'buy',
  'love',
  'love',
  'love',
  'this',
  'jumpsuit',
  'it',
  'fun',
  'flirty',
  'and',
  'fabulous',
  'every',
  'time',
  'wear',
  'it',
  'get',
  'nothing',
  'but',
  'great',
  'compliments',
  'Pants'],
 ['Flattering',
  'shirt',
  'This',
  'shirt',
  'is',
  'very',
  'flattering',
  'to',
  'all',
  'due',
  'to',
  'the',
  'adjustable',
  'front',
  'tie',
  'it',
  'is',
  'the',
  'perfect',
  'length',
  'to',
  'wear',
  'with',
  'leggings',
  'and',
  'it',
  'is',
  'sleeveless',
  'so',
  'it',
  'pairs',
  'well',
  'with',
  'any',
  'cardigan',
  'love',
  'this',
  'shirt',
  'Blouses']]

In [54]:
df_train_content = pd.DataFrame({'train_content_S':train_content_S})
df_test_content = pd.DataFrame({'test_content_S':test_content_S})
print(df_train_content.head(5))
print(df_test_content.head(5))

                                     train_content_S
0  [Absolutely, wonderful, silky, and, sexy, and,...
1  [Love, this, dress, it, sooo, pretty, happened...
2  [Some, major, design, flaws, had, such, high, ...
3  [Not, for, the, very, petite, love, tracy, ree...
4  [Cagrcoal, shimmer, fun, aded, this, in, my, b...
                                      test_content_S
0  [My, favorite, buy, love, love, love, this, ju...
1  [Flattering, shirt, This, shirt, is, very, fla...
2  [Such, fun, dress, and, lbs, ordered, the, pet...
3  [This, dress, is, perfection, so, pretty, and,...
4  [Runs, big, Bought, the, black, xs, to, go, un...


In [55]:
stopwords = pd.read_csv(r"stop_words.csv")
stopwords.head()

,stopword
0,!
1,""""
2,#
3,$
4,%


In [56]:
def drop_stopwords(contents, stopwords):
    contents_clean = []
    all_words = []
    
    for line in contents:
        line_clean = []
        
        for word in line:
            if word in stopwords:
                continue
                
            line_clean.append(word)
            all_words.append(word)
            
        contents_clean.append(line_clean)
    return contents_clean, all_words

In [57]:
train_contents = df_train_content.train_content_S.values.tolist()
test_contents_clean = df_test_content.test_content_S.values.tolist()
stopwords = stopwords.stopword.values.tolist()
train_contents_clean,train_all_words = drop_stopwords(train_contents, stopwords)
test_contents_clean,test_all_words = drop_stopwords(test_contents_clean, stopwords)

In [58]:
df_train_content = pd.DataFrame({'train_contents_clean':train_contents_clean})
df_test_content = pd.DataFrame({'test_contents_clean':test_contents_clean})
print(df_train_content.head(5))
print(df_test_content.head(5))

                                train_contents_clean
0  [Absolutely, wonderful, silky, sexy, comfortab...
1  [Love, dress, sooo, pretty, happened, find, st...
2  [Some, major, design, flaws, high, hopes, dres...
3  [Not, petite, love, tracy, reese, dresses, pet...
4  [Cagrcoal, shimmer, fun, aded, basket, hte, mi...
                                 test_contents_clean
0  [My, favorite, buy, love, love, love, jumpsuit...
1  [Flattering, shirt, This, shirt, flattering, d...
2  [Such, fun, dress, lbs, ordered, petite, make,...
3  [This, dress, perfection, pretty, flattering, ...
4  [Runs, big, Bought, black, xs, larkspur, midi,...


In [61]:
df_train_all_words = pd.DataFrame({'train_all_words':train_all_words})
df_test_all_words = pd.DataFrame({'test_all_words':test_all_words})
df_all_words = pd.concat([df_train_all_words['train_all_words'],df_test_all_words['test_all_words']],axis = 0).values.tolist()
# print(df_train_all_words.head())
# print(df_test_all_words.head())
print (df_all_words[0:5])

['Absolutely', 'wonderful', 'silky', 'sexy', 'comfortable']


In [62]:
train_contents_clean_list = []
for i in range(len(train_contents_clean)):
    train_contents_clean_list.append(" ".join(train_contents_clean[i]))
# test_contents_clean_list
# len(df_train_content)
df_train = pd.DataFrame({'train_contents_clean':train_contents_clean_list,'label':train['Recommended IND']})
df_train

,train_contents_clean,label
0,Absolutely wonderful silky sexy comfortable In...,1
1,Love dress sooo pretty happened find store gla...,1
2,Some major design flaws high hopes dress wante...,0
3,Not petite love tracy reese dresses petite fee...,0
4,Cagrcoal shimmer fun aded basket hte mintue pe...,1
...,...,...
18784,Great dress occasions happy snag dress great p...,1
18785,Wish made cotton It reminds maternity clothes ...,1
18786,Cute This fit top worked glad store didn order...,0
18787,Very cute dress perfect summer parties bought ...,1


In [63]:
test_contents_clean_list = []
for i in range(len(test_contents_clean)):
    test_contents_clean_list.append(" ".join(test_contents_clean[i]))
# test_contents_clean_list
# len(df_train_content)
df_test = pd.DataFrame({'test_contents_clean':test_contents_clean_list,'label':test['Recommended IND']})
df_test

,test_contents_clean,label
0,My favorite buy love love love jumpsuit fun fl...,1
1,Flattering shirt This shirt flattering due adj...,1
2,Such fun dress lbs ordered petite make length ...,1
3,This dress perfection pretty flattering Dresses,1
4,Runs big Bought black xs larkspur midi dress d...,1
...,...,...
4692,Comfortable super flattering store fit good le...,1
4693,Dresses,1
4694,Much person Yes great dress wasn online color ...,1
4695,Entrancing impressed beautiful color combinati...,1


In [66]:
df_train = pd.DataFrame({'train_words_list':train_contents_clean_list,'label':train['Recommended IND']})
df_test = pd.DataFrame({'test_words_list':test_contents_clean_list})
df_test.head()

,test_words_list
0,My favorite buy love love love jumpsuit fun fl...
1,Flattering shirt This shirt flattering due adj...
2,Such fun dress lbs ordered petite make length ...
3,This dress perfection pretty flattering Dresses
4,Runs big Bought black xs larkspur midi dress d...


In [67]:
df_train.label.unique()

array([1, 0], dtype=int64)

In [68]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer =  TfidfVectorizer(stop_words= stopwords, use_idf=True, smooth_idf=True, norm=None, max_df =0.7)
train_idf = vectorizer.fit_transform(df_train.train_words_list.values.tolist())
pre_idf =  vectorizer.transform(df_test.test_words_list.values.tolist())

In [69]:
train_idf

<18789x12346 sparse matrix of type '<class 'numpy.float64'>'
	with 416204 stored elements in Compressed Sparse Row format>

In [70]:
pre_idf

<4697x12346 sparse matrix of type '<class 'numpy.float64'>'
	with 101994 stored elements in Compressed Sparse Row format>

In [71]:
from sklearn.model_selection import train_test_split
# attr = df_train['train_words_list']
train_label = df_train['label']
x_train,x_test,y_train,y_test = train_test_split(train_idf,train_label.values.tolist(),test_size=0.2,random_state = 1,stratify = train_label)

In [72]:
x_train

<15031x12346 sparse matrix of type '<class 'numpy.float64'>'
	with 332748 stored elements in Compressed Sparse Row format>

In [73]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB(alpha=0.95)
classifier.fit(x_train,y_train)

MultinomialNB(alpha=0.95)

In [74]:
y_pred = classifier.predict(x_test)

In [75]:
from sklearn.metrics import precision_score,recall_score,f1_score
print(f"朴素贝叶斯-precision : {round(precision_score(y_test,y_pred)*100,2)}%")
print(f"朴素贝叶斯-recall : {round(recall_score(y_test,y_pred)*100,2)}%")
print(f"朴素贝叶斯-f1 : {round(f1_score(y_test,y_pred)*100,2)}%")

朴素贝叶斯-precision : 94.85%
朴素贝叶斯-recall : 86.96%
朴素贝叶斯-f1 : 90.73%


In [76]:
# y_pred_test = classifier.predict_proba(pre_idf)
y_pred_test = classifier.predict(pre_idf)

In [77]:
len(y_pred_test)
ID_test= test['CID']
# len(ID_test)
final_result=pd.DataFrame({'CID':ID_test,'Predicted_Results':y_pred_test})

4697

In [78]:
final_result.Predicted_Results.value_counts()

1    3612
0    1085
Name: Predicted_Results, dtype: int64

In [79]:
final_result.to_csv('Final_Results.csv',index =None)

In [80]:
classifier = MultinomialNB(alpha=0.95)
classifier.fit(train_idf,train_label)

MultinomialNB(alpha=0.95)

In [82]:
y_pred = classifier.predict(pre_idf)
print(f"朴素贝叶斯-precision : {round(precision_score(test_label,y_pred)*100,2)}%")
print(f"朴素贝叶斯-recall : {round(recall_score(test_label,y_pred)*100,2)}%")
print(f"朴素贝叶斯-f1 : {round(f1_score(test_label,y_pred)*100,2)}%")

朴素贝叶斯-precision : 95.61%
朴素贝叶斯-recall : 89.08%
朴素贝叶斯-f1 : 92.23%
